# CardID Extractor — Colab GPU

از Runtime → Change runtime type یک GPU مثل T4 انتخاب کن.
این نمونه پکیج را در محیط مجازی مستقل نصب می‌کند تا با Gradio/Diffusers محیط اصلی تعارض نداشته باشد.
وزن مدل‌ها در اولین اجرا دانلود می‌شوند. این نوت‌بوک از subprocess محیط مجازی استفاده می‌کند؛ kernel اصلی عوض نمی‌شود.
برای استفاده در سرویس، تابع `CardExtractor.extract` را در همان محیط برنامهٔ خودت فراخوانی کن؛ مثال `use_in_service.py` داخل بسته است.


In [ ]:
# Upload the source ZIP delivered with this package.
from google.colab import files
from pathlib import Path
import zipfile
import sys
import subprocess

archives = files.upload()
archive_names = [name for name in archives if name.endswith('.zip')]
if len(archive_names) != 1:
    raise ValueError('Choose exactly one source ZIP: cardid-extractor-0.1.0.zip')
source_dir = Path('/content/cardid-source')
source_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(archive_names[0]) as archive:
    for member in archive.infolist():
        resolved = (source_dir / member.filename).resolve()
        if not resolved.is_relative_to(source_dir.resolve()):
            raise ValueError('Archive member is outside the source directory')
    archive.extractall(source_dir)
projects = list(source_dir.glob('*/pyproject.toml'))
if len(projects) != 1:
    raise RuntimeError('Could not locate the source package')
PROJECT = projects[0].parent
print('Project:', PROJECT)


In [ ]:
# Isolated installation. CUDA torch/torchvision and ML dependencies remain in this venv.
subprocess.run([sys.executable, str(PROJECT / 'scripts/setup_colab.py')], check=True)
PYTHON = '/content/cardid-venv/bin/python'


## عکس‌های ورودی

یک یا چند عکس کارت را انتخاب کن. گزینهٔ `TEST_GRAYSCALE` هر عکس را یک بار دیگر به‌صورت grayscale آزمایش می‌کند.
اجرای همهٔ عکس‌ها در یک process انجام می‌شود و مدل‌ها بین آن‌ها دوباره بارگذاری نمی‌شوند.


In [ ]:
photos = files.upload()
input_dir = Path('/content/cardid-inputs')
input_dir.mkdir(exist_ok=True)
INPUTS = []
for name, data in photos.items():
    destination = input_dir / Path(name).name
    destination.write_bytes(data)
    INPUTS.append(str(destination))
if not INPUTS:
    raise ValueError('Upload at least one image')
TEST_GRAYSCALE = False


In [ ]:
RESULTS = Path('/content/cardid-results')
# A fresh output directory avoids displaying an earlier crop after a failed retry.
from datetime import datetime
RESULTS = RESULTS / datetime.now().strftime('run_%Y%m%d_%H%M%S_%f')
command = [PYTHON, str(PROJECT / 'examples/smoke_gpu.py'), *INPUTS,
           '--output-dir', str(RESULTS), '--cache-dir', '/content/cardid-model-cache']
if TEST_GRAYSCALE:
    command.append('--also-gray')
completed = subprocess.run(command)
if completed.returncode not in (0, 2):
    raise RuntimeError('Extraction failed; see the output above')
if completed.returncode == 2:
    print('Some inputs need review or no card was found. See JSON reports.')


In [ ]:
from IPython.display import display, Image as DisplayImage
for output in sorted(RESULTS.glob('*.png')):
    print(output.name)
    display(DisplayImage(filename=str(output)))

import shutil
result_zip = shutil.make_archive(str(RESULTS), 'zip', RESULTS)
files.download(result_zip)
